Criação de um ficheiro CSV com a informação relativa a nº de deglutiçoes, tempo do ficheiro .wav e duração do evento detetato

In [2]:
!pip install tabulate

In [ ]:
import numpy as np
import os
import glob
import scipy.io.wavfile as wav
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.signal import find_peaks

In [ ]:
def short_time_energy(signal, frame_size, hop_size):
    """
    Compute Short-Time Energy (STE) of the input signal.
    """
    energy = []
    for i in range(0, len(signal) - frame_size, hop_size):
        frame = signal[i : i + frame_size]
        energy.append(np.sum(frame ** 2))
    return np.array(energy)

# -------------------
# 1. Input Folder & List Audio Files
# -------------------
folder_path = r"C:\Users\Teresa\Desktop\MBBAS (2ºano)\Tese\Audio_samples"
audio_files = glob.glob(os.path.join(folder_path, "*.wav"))

if not audio_files:
    print("No WAV files found in the folder.")
else:
    for file_path in audio_files:
        print(f"\nProcessing file: {os.path.basename(file_path)}")
        
        # -------------------
        # 2. Load and Preprocess
        # -------------------
        sample_rate, audio = wav.read(file_path)
        
        # If stereo, convert to mono
        if len(audio.shape) > 1:
            audio = np.mean(audio, axis=1)
        
        # Normalize
        audio = audio / np.max(np.abs(audio))
        
        # -------------------
        # 3. Compute STE and Setup Parameters
        # -------------------
        frame_duration = 0.05  # 50 ms
        hop_duration = 0.01    # 10 ms
        
        frame_size = int(frame_duration * sample_rate)
        hop_size = int(hop_duration * sample_rate)
        
        ste = short_time_energy(audio, frame_size, hop_size)
        
        # -------------------
        # 4. Compute Dynamic Threshold (Moving Window)
        # -------------------
        window_size = 100  # Number of frames in the moving window
        
        thresholds = np.array([
            np.mean(ste[max(0, i-window_size):i+window_size]) + 1.5 * np.std(ste[max(0, i-window_size):i+window_size])
            for i in range(len(ste))
        ])
        
        # -------------------
        # 5. Detect Peaks in STE
        # -------------------
        min_distance = int(0.5 / hop_duration)
        
        peaks, properties = find_peaks(ste, height=thresholds, distance=min_distance, prominence=0.1)
        
        # Convert peak indices to time in seconds
        peak_times = peaks * hop_duration
        
        print(f"Number of detected peaks: {len(peaks)}")
        print("Peak times (s):", peak_times)
        
        # -------------------
        # 6. Print Time Windows
        # -------------------
        window_len = 0.3  # e.g., +/- 0.3s around each detected peak
        for i, peak_time in enumerate(peak_times):
            start_time = max(peak_time - window_len, 0.0)
            end_time = min(peak_time + window_len, len(audio)/sample_rate)
            print(f"Swallow {i+1}: from {start_time:.2f}s to {end_time:.2f}s")
        
        # -------------------
        # 7. Plot Waveform & STE with Dynamic Threshold
        # -------------------
        sns.set(style="darkgrid")
        fig, axs = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
        
        # Time axes for plotting
        time_waveform = np.linspace(0, len(audio) / sample_rate, len(audio))
        time_ste = np.linspace(0, len(audio) / sample_rate, len(ste))
        
        # Plot raw waveform
        axs[0].plot(time_waveform, audio, color='blue', alpha=0.6)
        axs[0].set_title(f"Waveform: {os.path.basename(file_path)}")
        axs[0].set_ylabel("Amplitude")
        
        # Plot STE and dynamic threshold
        axs[1].plot(time_ste, ste, color='red', label="Short-Time Energy")
        axs[1].plot(time_ste[peaks], ste[peaks], "x", color='black', label="Detected Peaks")
        axs[1].plot(time_ste, thresholds, color='green', linestyle='dashed', label="Dynamic Threshold")
        axs[1].set_title("Short-Time Energy with Dynamic Threshold & Detected Swallows")
        axs[1].set_xlabel("Time (s)")
        axs[1].set_ylabel("Energy")
        axs[1].legend()
        
        plt.tight_layout()
        plt.show()


#### Dynamic threshold values, hop and frame size